# EV Charging Dynamic Tariff Optimization System

This notebook orchestrates the end-to-end simulation pipeline.

In [ ]:
import os
import sys
# Add parent directory to path to import src and config
sys.path.append(os.path.abspath('..'))

from main import run_simulation
run_simulation()

# EV Charging Dynamic Tariff Optimization System
This notebook provides a full analysis and simulation of the EV Charging Dynamic Tariff Optimization pipeline.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))

from config.settings import STATIONS_CSV_PATH, ACNDATA_JSON_PATH
from src.data_loader import load_stations_data, load_sessions_data
from src.preprocessing import preprocess_sessions, aggregate_hourly
from src.models import DemandPredictionAgent, TariffPricingAgent, MonitoringLearningAgent
from src.evaluator import compute_kpis, plot_simulation_results

print("✅ All imports successful!")

## 1. Load Data

In [ ]:
# Load stations
stations_df = load_stations_data()
print(f"✅ Stations loaded: {stations_df.shape}")
stations_df.head()

In [ ]:
# Load sessions
sessions_df = load_sessions_data()
print(f"✅ Sessions loaded: {sessions_df.shape}")
sessions_df.head()

### Dataset Note
Two datasets were identified for this project:
1. **ACN-Data** — used as the primary dataset for session-level 
   modelling due to its granular timestamps, energy delivery records, 
   and station metadata required for tariff optimization.
2. **UrbanEV (ST-EVCDP)** — explored for spatial demand patterns 
   and peak-hour analysis. UrbanEV insights informed the off-peak 
   threshold selection (30%) and surge pricing trigger (80%) used 
   in the Tariff Pricing Agent.

Primary modelling uses ACN-Data as it contains the session-level 
granularity required for the agentic pricing pipeline.

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print("Sessions Info:")
print(sessions_df.info())
print("\nBasic Statistics:")
sessions_df.describe()

In [ ]:
# Session count summary for EDA
print("=" * 50)
print("Dataset Overview")
print("=" * 50)
print(f"Total sessions       : {len(sessions_df):,}")
print(f"Date range start     : {sessions_df['connectionTime'].min()}")
print(f"Date range end       : {sessions_df['connectionTime'].max()}")
print(f"Unique stations      : {sessions_df['stationID'].nunique()}")
print(f"Unique sites         : {sessions_df['siteID'].nunique()}")
print(f"Avg kWh per session  : {sessions_df['kWhDelivered'].mean():.2f} kWh")
print(f"Total kWh delivered  : {sessions_df['kWhDelivered'].sum():,.2f} kWh")
print(f"Median session kWh   : {sessions_df['kWhDelivered'].median():.2f} kWh")

In [ ]:
print("Missing Values:")
missing = sessions_df.isnull().sum()
print(missing[missing > 0])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Energy delivered distribution
axes[0,0].hist(sessions_df['kWhDelivered'].dropna(), bins=50, color='steelblue', edgecolor='black')
axes[0,0].set_title('Distribution of Energy Delivered (kWh)')
axes[0,0].set_xlabel('kWh Delivered')
axes[0,0].set_ylabel('Count')

# Plot 2: Sessions per site
if 'siteID' in sessions_df.columns:
    sessions_df['siteID'].value_counts().plot(kind='bar', ax=axes[0,1], color='coral')
    axes[0,1].set_title('Sessions per Site')
    axes[0,1].set_xlabel('Site ID')
    axes[0,1].set_ylabel('Number of Sessions')

# Plot 3: Connection duration
if 'connectionTime' in sessions_df.columns and 'disconnectTime' in sessions_df.columns:
    # sessions_df['connectionTime'] = pd.to_datetime(sessions_df['connectionTime'], utc=True)
    # sessions_df['disconnectTime'] = pd.to_datetime(sessions_df['disconnectTime'], utc=True)
    sessions_df['duration_hrs'] = (sessions_df['disconnectTime'] - sessions_df['connectionTime']).dt.total_seconds() / 3600
    axes[1,0].hist(sessions_df['duration_hrs'].clip(0, 24).dropna(), bins=50, color='green', edgecolor='black')
    axes[1,0].set_title('Session Duration Distribution (hrs)')
    axes[1,0].set_xlabel('Duration (hours)')
    axes[1,0].set_ylabel('Count')

# Plot 4: Sessions over time
if 'connectionTime' in sessions_df.columns:
    sessions_df['date'] = sessions_df['connectionTime'].dt.date
    daily_counts = sessions_df.groupby('date').size()
    daily_counts.plot(ax=axes[1,1], color='purple')
    axes[1,1].set_title('Daily Charging Sessions Over Time')
    axes[1,1].set_xlabel('Date')
    axes[1,1].set_ylabel('Number of Sessions')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved!")

## 3. Preprocessing & Hourly Aggregation

In [ ]:
# Fix datetime columns before aggregation
datetime_cols = ['connectionTime', 'disconnectTime', 'doneChargingTime']

for col in datetime_cols:
    if col in sessions_df.columns:
        sessions_df[col] = pd.to_datetime(sessions_df[col], utc=True, errors='coerce')
        print(f"✅ {col} converted: {sessions_df[col].dtype}")

# Now run aggregation
df_hourly = aggregate_hourly(sessions_df)
print(f"✅ Hourly aggregation done: {df_hourly.shape}")
df_hourly.head()

## 4. Model Training & Tariff Optimization

In [ ]:
# Split data
eval_steps = 752
df_train = df_hourly.iloc[:-eval_steps].copy()
df_eval = df_hourly.iloc[-eval_steps:].copy()
print(f"✅ Train size: {df_train.shape}, Eval size: {df_eval.shape}")

# Train Demand Agent
demand_agent = DemandPredictionAgent()
demand_agent.fit(df_train)
print("✅ Demand agent trained!")

# Tariff Agent
tariff_agent = TariffPricingAgent()

# MonitoringAgent
monitor = MonitoringLearningAgent()

# Loop through each eval row and generate predictions + tariffs
results = []
for _, row in df_eval.iterrows():
    dow = int(row['hour_dt'].dayofweek) if 'hour_dt' in df_eval.columns else int(row.name.dayofweek)
    hr  = int(row['hour_dt'].hour) if 'hour_dt' in df_eval.columns else int(row.name.hour)

    # Predict demand
    pred = demand_agent.predict(dayofweek=dow, hour=hr)

    # Calculate tariff
    tariff = tariff_agent.calculate_tariff(
        predicted_kwh=pred['predicted_kwh'],
        avg_kwh=demand_agent.global_mean_kwh
    )

    results.append({
        'hour_dt': row.get('hour_dt', row.name),
        'predicted_kwh': pred['predicted_kwh'],
        'tariff': tariff,
        'actual_kwh': row.get('total_kwh', np.nan)
    })

results_df = pd.DataFrame(results)
print(f"✅ Predictions & tariffs generated: {len(results_df)} steps")
results_df.head()

### Model Performance Note
R² of 0.293 is expected for a rule-based lookup model on 
high-variance real-world EV data. The model captures temporal 
baseline patterns (day-of-week × hour-of-day) but underestimates 
demand spikes.

- Global mean kWh per session: **8.98 kWh**
- MAE of 17.59 kWh indicates overestimation in low-activity hours
- RMSE of 27.47 kWh reflects sensitivity to peak demand spikes

**Future improvement**: Replace lookup table with XGBoost or LSTM 
for improved demand forecasting accuracy.

## 5. Simulation & KPIs

In [ ]:
with open('c:/projects_antig/src/evaluator.py', 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines, 1):
    print(f"{i}: {line}", end='')

In [ ]:
# Build kpi_df with exact column names evaluator.py expects
kpi_df = results_df.copy()
kpi_df['actual_kwh'] = df_eval['kwh_demand'].values
kpi_df = kpi_df.rename(columns={'tariff': 'tariff_price'})

# Add utilization = actual / predicted (capped at 1)
kpi_df['utilization'] = (
    kpi_df['actual_kwh'] / kpi_df['predicted_kwh'].replace(0, np.nan)
).clip(0, 1).fillna(0)

# Add satisfaction = inverse of price deviation from base rate
kpi_df['satisfaction'] = (
    1 - ((kpi_df['tariff_price'] - 0.25) / 0.25).abs()
).clip(0, 1)

print("kpi_df columns:", kpi_df.columns.tolist())

# Compute KPIs
kpis = compute_kpis(kpi_df)
print("\n Key Performance Indicators:")
for k, v in kpis.items():
    print(f"  {k}: {v}")

In [ ]:
# ₹ INR Revenue Comparison (as per project brief ₹15/kWh baseline)
base_price = 15
usd_to_inr = 83
flat_revenue = kpi_df['actual_kwh'].sum() * base_price
dynamic_revenue = (kpi_df['actual_kwh'] * kpi_df['tariff_price'] * usd_to_inr).sum()
revenue_gain = ((dynamic_revenue - flat_revenue) / flat_revenue) * 100
print(f"Revenue Gain % vs ₹15/kWh baseline: {revenue_gain:.2f}%")

print("=" * 50)
print("Revenue Comparison vs ₹15/kWh Brief Baseline")
print("=" * 50)
print(f"Flat Revenue  (₹15/kWh)     : ₹{flat_revenue_inr:,.2f}")
print(f"Dynamic Revenue (converted) : ₹{dynamic_revenue_inr:,.2f}")
print(f"Revenue Gain vs ₹15 baseline: {revenue_gain_inr:.2f}%")
print()
print("Note: Dynamic tariffs computed in USD ($0.25 base)")
print("Converted to INR at 1 USD = ₹83 for comparison")

In [ ]:
# Satisfaction Metric Explanation
print("""
Satisfaction Metric Methodology:
─────────────────────────────────────────────────
Formula : 1 - abs(tariff_price - base_price) / base_price
Capped  : between 0 and 1
Result  : 96.3% average satisfaction

Interpretation:
Dynamic tariffs stayed within ~3.7% of the $0.25 base
rate on average — meaning users experienced minimal
price deviation from what they would pay on a flat rate.

Limitation: This is a proxy metric derived from tariff
deviation. It does not represent direct user feedback
or survey data.
─────────────────────────────────────────────────
""")

In [ ]:
# Pricing Efficiency Score (Revenue per kWh delivered)
pricing_efficiency = (kpi_df['actual_kwh'] * kpi_df['tariff_price']).sum() / kpi_df['actual_kwh'].sum()
flat_efficiency = 0.25

print("=" * 50)
print("Pricing Efficiency Score")
print("=" * 50)
print(f"Dynamic Pricing  : ${pricing_efficiency:.4f} revenue per kWh")
print(f"Flat Rate Baseline: ${flat_efficiency:.4f} revenue per kWh")
print(f"Efficiency Gain  : ${pricing_efficiency - flat_efficiency:.4f}/kWh")
print()
print("Pricing efficiency tracks whether the feedback")
print("loop is improving revenue decisions over time.")

In [ ]:
import os
os.makedirs('notebooks', exist_ok=True)
plot_simulation_results(kpi_df, save_path='notebooks/simulation_report.png')

## 6. Full Simulation Run

In [ ]:
from main import run_simulation
run_simulation()
print("✅ Full simulation complete!")

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(kpi_df['actual_kwh'], kpi_df['predicted_kwh'])
print(f"R² Score: {r2}")

In [ ]:
base_price = 15  # ₹15/kWh as per project requirement
flat_revenue = kpi_df['actual_kwh'].sum() * base_price
dynamic_revenue = (kpi_df['actual_kwh'] * kpi_df['tariff_price']).sum()
revenue_gain = ((dynamic_revenue - flat_revenue) / flat_revenue) * 100
print(f"Revenue Gain % vs ₹15/kWh baseline: {revenue_gain:.2f}%")

In [ ]:
off_peak = kpi_df[kpi_df['utilization'] < 0.30]
print(f"Off-peak hours: {len(off_peak)}")
print(f"Avg tariff in off-peak: {off_peak['tariff_price'].mean():.4f}")

In [ ]:
# import os
# import pandas as pd

# # Create folder if it doesn't exist
# os.makedirs('c:/projects_antig/dataset', exist_ok=True)

# # Save KPI results as CSV
# kpi_df.to_csv('c:/projects_antig/dataset/simulation_results.csv', index=False)

# kpi_summary = pd.DataFrame([kpis])
# kpi_summary.to_csv('c:/projects_antig/dataset/kpi_summary.csv', index=False)

# print("✅ CSVs saved!")